# Gizli Dirichlet Dağılımı (Latent Dirichlet Allocation) (LDA)

🎯 Bu challenge'ın amacı, **LDA** algoritması (NLP'de Denetimsiz Öğrenme) ile e-posta külliyatı içinde konular bulmaktır.

✉️ İşte 1000'den fazla ***etiketlenmemiş e-posta*** içeren bir koleksiyon. Bunlardan ***konuları çıkarmaya*** çalışalım!

In [1]:
import pandas as pd

url = 'https://d32aokrjazspmn.cloudfront.net/materials/lda_data'

data = pd.read_csv(url, sep=",", header=None)
data.columns = ['text']
data.head()

,text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...


In [2]:
data.shape

(1199, 1)

## (1) Preprocessing 

❓ **Question (Cleaning**) ❓ You're used to it by now... Clean up! Store the cleaned text in a new column "clean_text" of the DataFrame.

In [5]:
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))

def clean_for_lda(text):

    text = text.lower()

    text = re.sub(r'\d+', '', text)

    text = text.translate(str.maketrans('', '', string.punctuation))

    tokens = word_tokenize(text)

    lemmatizer = WordNetLemmatizer()
    cleaned_tokens = [
        lemmatizer.lemmatize(w) for w in tokens
        if w not in stop_words and len(w) > 2
    ]

    return " ".join(cleaned_tokens)

data['clean_text'] = data['text'].apply(clean_for_lda)

print(data[['text', 'clean_text']].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/fukansimsek/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/fukansimsek/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/fukansimsek/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


                                                text  \
0  From: gld@cunixb.cc.columbia.edu (Gary L Dare)...   
1  From: atterlep@vela.acs.oakland.edu (Cardinal ...   
2  From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...   
3  From: atterlep@vela.acs.oakland.edu (Cardinal ...   
4  From: vzhivov@superior.carleton.ca (Vladimir Z...   

                                          clean_text  
0  gldcunixbcccolumbiaedu gary dare subject stan ...  
1  atterlepvelaacsoaklandedu cardinal ximenez sub...  
2  minerkuhubccukansedu subject ancient book orga...  
3  atterlepvelaacsoaklandedu cardinal ximenez sub...  
4  vzhivovsuperiorcarletonca vladimir zhivov subj...  


## (2) Latent Dirichlet Allocation model

❓ **Soru (Eğitim)** ❓ Potansiyel konuları çıkarmak için bir LDA modeli eğitin

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vectorizer = TfidfVectorizer(max_df=0.95, min_df=2)
data_vectorized = vectorizer.fit_transform(data['clean_text'])

lda_model = LatentDirichletAllocation(n_components=5, random_state=42)
lda_model.fit(data_vectorized)

print("LDA modeli başarıyla eğitildi!")

LDA modeli başarıyla eğitildi!


##  (3) Potansiyel konuları görselleştirin

🎁 Potansiyel konularla ilişkili kelimeleri yazdırmak için bir  fonksiyon kodladık.

In [8]:
def print_topics(model, vectorizer):
    for idx, topic in enumerate(model.components_):
        print("Topic %d:" % (idx))
        print([(vectorizer.get_feature_names_out()[i], topic[i])
                        for i in topic.argsort()[:-10 - 1:-1]])

❓ **Soru** ❓ LDA tarafından çıkarılan konuları yazdırın.

In [10]:
print_topics(lda_model, vectorizer)

Topic 0:
[('grass', 4.300425615532723), ('valley', 3.9878379924179734), ('gainey', 3.88969756208158), ('european', 3.8132138436053395), ('petchgvggvgtekcom', 3.2363889341382492), ('ahl', 2.7663729762050675), ('sweden', 2.616891629416868), ('daily', 2.479960914314467), ('chuck', 2.4200804338494875), ('petch', 2.299644232079939)]
Topic 1:
[('mask', 5.391112547065467), ('valerie', 1.3785506322997292), ('hammerl', 1.3785506322994114), ('hammerlacsubuffaloedu', 1.378522164475149), ('finalswinner', 1.3643022019558566), ('rolfe', 1.217422142837256), ('holger', 1.1683590244262134), ('finalswho', 1.1538264823534115), ('statemaine', 1.1538264823534112), ('lazarus', 1.144358805462673)]
Topic 2:
[('captain', 8.308167041706037), ('dare', 5.9840248281073825), ('abc', 5.802373268422094), ('gary', 5.72169599806296), ('espn', 5.303580257364541), ('period', 5.048359071293524), ('gldcunixbcccolumbiaedu', 4.502316107434116), ('chi', 4.306611560629978), ('pt', 4.222528740146613), ('cal', 4.038692919640248)

## (4) Yeni bir metnin belge-konu karışımını tahmin edin

❓ **Soru (Tahmin)** ❓

LDA modeliniz fit edildiğine göre, onu yeni bir metnin konularını tahmin etmek için kullanabilirsiniz.

1. Örneği vektörleştirin
2. Vektörleştirilmiş örnek üzerinde LDA'yı kullanarak konuları tahmin edin

In [12]:
example = ["My team performed poorly last season. Their best player was out injured and only played one game"]

In [13]:
example_vectorized = vectorizer.transform(example)

topic_probabilities = lda_model.transform(example_vectorized)

print("Konu Olasılıkları:")
print(topic_probabilities)

import numpy as np
predicted_topic = np.argmax(topic_probabilities)
print(f"\nBu metin en yüksek ihtimalle Konu #{predicted_topic + 1} ile ilgili.")

Konu Olasılıkları:
[[0.04948345 0.04947794 0.04955336 0.80200421 0.04948104]]

Bu metin en yüksek ihtimalle Konu #4 ile ilgili.


🏁 Tebrikler! LDA'yı hızlı bir şekilde nasıl uygulayacağınızı öğrendiniz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

🚀 ... ve bir sonraki göreve geçin!